# Bronze -> Silver: visão consolidada

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Volumetria, completude e o que a limpeza Silver muda em relação à Bronze, para as 4 tabelas Bronze +
6 Silver de uma vez -- estes dois estágios são ingestão/limpeza, não modelagem, então não têm
profundidade individual suficiente para justificar um notebook por tabela (ver ADR 0022, opções
consideradas).

`ugc_mentions` (Bronze e Silver) entrou aqui depois -- é uma coleção independente (actor/cadência
próprios via `scripts/run_ugc_mentions.py`, ADR 0020 Ficha 8 / issue #93), sem par direto de
`profiles`/`posts`/`reels`, mas cabe no mesmo notebook consolidado pelo mesmo raciocínio: pouca
profundidade analítica própria de ingestão/limpeza (o domínio de UGC que merece profundidade fica
em `gold_ugc.ipynb`).

In [1]:
import sys
import os

# Adiciona o diretório raiz do projeto ao sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

from config import settings
from src.data_extract.bronze_writer import BronzeWriter
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import completeness_summary, count_duplicate_rows

load_dotenv()
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

## Carga

Bronze via `BronzeWriter.get_latest_*` -- o mesmo acesso que `pipeline.py` usa para checar se já há
dado local (`_bronze_has_data`). `DeltaRepository` cobre `posts_clean`/`reels_clean`/
`governors_metadata`, mas não expõe `profiles_clean`/`comments_clean` diretamente (`load_profiles()`
do repositório na verdade lê `governor_engagement`, já em Gold -- nome legado do repositório, não
confundir); essas duas leem a tabela Delta diretamente pelo path de `settings`, mesmo primitivo que
`DeltaRepository._load` usa por baixo.

In [2]:
bronze = BronzeWriter(
    bronze_profiles_path=settings.BRONZE_PROFILES,
    bronze_posts_path=settings.BRONZE_POSTS,
    bronze_reels_path=settings.BRONZE_REELS,
    bronze_ugc_mentions_path=settings.BRONZE_UGC_MENTIONS,
)
repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

tabelas = {
    'bronze.instagram_profiles': bronze.get_latest_profiles(),
    'bronze.instagram_posts': bronze.get_latest_posts(),
    'bronze.instagram_reels': bronze.get_latest_reels(),
    'silver.profiles_clean': DeltaTable(str(settings.SILVER_PROFILES)).to_pandas(),
    'silver.posts_clean': repo.load_posts(),
    'silver.reels_clean': repo.load_reels(),
    'silver.comments_clean': DeltaTable(str(settings.SILVER_COMMENTS)).to_pandas(),
    'silver.governors_metadata': repo.load_governors_metadata(),
    'bronze.ugc_mentions': bronze.get_latest_ugc_mentions(),
    'silver.ugc_mentions_clean': DeltaTable(str(settings.SILVER_UGC_MENTIONS)).to_pandas(),
}

## Volumetria e duplicatas

In [3]:
resumo = pd.DataFrame({
    'tabela': list(tabelas.keys()),
    'n_linhas': [len(df) for df in tabelas.values()],
    'n_colunas': [df.shape[1] for df in tabelas.values()],
    'n_linhas_duplicadas': [count_duplicate_rows(df) for df in tabelas.values()],
})
resumo

,tabela,n_linhas,n_colunas,n_linhas_duplicadas
0,bronze.instagram_profiles,26,17,0
1,bronze.instagram_posts,230,17,0
2,bronze.instagram_reels,154,20,0
3,silver.profiles_clean,26,14,0
4,silver.posts_clean,228,17,0
5,silver.reels_clean,153,19,0
6,silver.comments_clean,1909,12,0
7,silver.governors_metadata,26,7,0
8,bronze.ugc_mentions,93,17,1
9,silver.ugc_mentions_clean,91,14,0


## Completude por tabela

Só colunas com pelo menos um nulo aparecem -- uma tabela sem saída abaixo do seu nome está
100% completa nas colunas que tem.

In [4]:
for nome, df in tabelas.items():
    resumo_completude = completeness_summary(df)
    colunas_com_nulo = resumo_completude[resumo_completude['n_nulos'] > 0]
    if not colunas_com_nulo.empty:
        print(f'--- {nome} ---')
        display(colunas_com_nulo)

--- bronze.instagram_profiles ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
3,businessCategoryName,object,10,38.46,3
7,postsCount,float64,1,3.85,25
8,igtvVideoCount,float64,9,34.62,17
12,hasChannel,object,26,100.00,0
13,joinedRecently,object,9,34.62,1


--- bronze.instagram_posts ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
0,id,object,2,0.87,228
1,ownerId,object,2,0.87,40
2,ownerUsername,object,2,0.87,40
4,commentsCount,float64,2,0.87,175
5,likesCount,float64,2,0.87,214
6,timestamp,object,2,0.87,226
7,type,object,2,0.87,3
8,shortCode,object,2,0.87,228
9,caption,object,2,0.87,219
10,videoViewCount,float64,82,35.65,148


--- bronze.instagram_reels ---


...

,coluna,dtype,n_nulos,pct_nulos,n_unicos
6,repliesCount,float64,1909,100.00,0


--- bronze.ugc_mentions ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
0,id,object,2,2.15,91
1,shortCode,object,2,2.15,91
2,type,object,2,2.15,3
3,caption,object,2,2.15,91
4,mentions,object,2,2.15,30
5,taggedUsers,object,2,2.15,85
6,likesCount,float64,2,2.15,76
7,commentsCount,float64,2,2.15,39
8,videoPlayCount,float64,55,59.14,38
9,timestamp,object,2,2.15,91


--- silver.ugc_mentions_clean ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
4,governor_username,object,1,1.10,20
9,videoPlayCount,float64,53,58.24,38


## O que a limpeza Silver muda em relação à Bronze

Para os três pares com correspondência direta (`profiles`, `posts`, `reels`): diferença de colunas
(o que a limpeza remove/deriva) e diferença de contagem de linhas (o que é descartado, ex.: perfis
sem dado suficiente para conformar o schema Silver).

In [5]:
pares_bronze_silver = {
    'profiles': ('bronze.instagram_profiles', 'silver.profiles_clean'),
    'posts': ('bronze.instagram_posts', 'silver.posts_clean'),
    'reels': ('bronze.instagram_reels', 'silver.reels_clean'),
    'ugc_mentions': ('bronze.ugc_mentions', 'silver.ugc_mentions_clean'),
}

linhas_comparacao = []
for entidade, (nome_bronze, nome_silver) in pares_bronze_silver.items():
    df_bronze = tabelas[nome_bronze]
    df_silver = tabelas[nome_silver]
    linhas_comparacao.append({
        'entidade': entidade,
        'linhas_bronze': len(df_bronze),
        'linhas_silver': len(df_silver),
        'linhas_descartadas': len(df_bronze) - len(df_silver),
        'colunas_removidas_pela_limpeza': sorted(set(df_bronze.columns) - set(df_silver.columns)),
        'colunas_derivadas_pela_limpeza': sorted(set(df_silver.columns) - set(df_bronze.columns)),
    })

pd.DataFrame(linhas_comparacao)

,entidade,linhas_bronze,linhas_silver,linhas_descartadas,colunas_removidas_pela_limpeza,colunas_derivadas_pela_limpeza
0,profiles,26,26,0,"[_source, hasChannel, igtvVideoCount, joinedRe...",[_source_layer]
1,posts,230,228,2,"[_source, locationName, timestamp, type, video...","[Tipo, _source_layer, data_hora, hashtags, typ..."
2,reels,154,153,1,"[_source, isPinned, latestComments, timestamp,...","[Tipo, Total de Engajamento, _source_layer, da..."
3,ugc_mentions,93,91,2,"[_source, mentions, ownerFullName, ownerId, ow...","[_source_layer, authorUsername, data_hora, gov..."


## Nota de interpretação

Comparar `linhas_descartadas` acima com o motivo de descarte esperado (perfil/post sem os campos
mínimos que `ProfileCleaner`/`PostCleaner` exigem -- ver `src/features/silver/`) antes de assumir que
uma queda grande é um bug de coleta: para um universo de ~27 governadores, descartar 1-3 linhas por
tabela costuma ser esperado (perfis novos sem posts suficientes, ou posts com campo crítico ausente
vindos do scraper), não um sinal de problema sistemático. Se `linhas_descartadas` crescer muito acima
disso numa execução futura, vale investigar a extração (Bronze) antes de suspeitar da limpeza
(Silver).

`ugc_mentions` é o único par onde `linhas_descartadas` não se compara ao universo de ~27
governadores -- o volume bruto depende da cadência do actor de terceiros
(`apify/instagram-tagged-scraper`), não do número de perfis monitorados. Além disso,
`governor_username` nulo na Silver não é um descarte (a linha permanece, só sem correlação
resolvida) -- ver completude e análise de covariáveis específica em `gold_ugc.ipynb`.